In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

# Hardcoded dummy data with proper date format
data = [
    ("541104", "84509G", "SET OF 4 FAIRY CAKE PLACEMATS", 3, "2011-01-13 10:15:00", 3.29, 12345, "United Kingdom"),
    ("541105", "21890", "S/6 WOODEN SKITTLES IN COTTON BAG", 5, "2011-01-15 14:30:00", 2.95, 17338, "United Kingdom"),
    ("541106", "84509G", "SET OF 4 FAIRY CAKE PLACEMATS", 2, "2011-01-20 09:45:00", 3.29, 54321, "France"),
    ("C541107", "21890", "S/6 WOODEN SKITTLES IN COTTON BAG", -2, "2011-02-10 11:20:00", 2.95, 17338, "United Kingdom"),
    ("541108", "21890", "S/6 WOODEN SKITTLES IN COTTON BAG", 10, "2011-02-15 16:00:00", 2.95, 67890, "Germany"),
    ("541109", "99999", "VINTAGE CARDBOARD SUITCASE", 1, "2011-02-20 13:10:00", 15.99, 11111, "United Kingdom")
]

# Define schema
schema = StructType([
    StructField("invoiceno", StringType(), True),
    StructField("stockcode", StringType(), True),
    StructField("description", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("invoicedate", StringType(), True),  # Keep as string first, then convert
    StructField("unitprice", DoubleType(), True),
    StructField("customerid", IntegerType(), True),
    StructField("country", StringType(), True)
])

# Create DataFrame
df = spark.createDataFrame(data, schema)

# Convert invoicedate string to proper timestamp
from pyspark.sql.functions import to_timestamp
df = df.withColumn("invoicedate", to_timestamp("invoicedate", "yyyy-MM-dd HH:mm:ss"))

# Create temporary view
df.createOrReplaceTempView("sales")

# Verify the data
df.display()


In [0]:
"""Find the best-selling item for each month (no need to separate months by year). The best-selling item is determined by the highest total sales amount, calculated as: total_paid = unitprice * quantity. A negative quantity indicates a return or cancellation (the invoice number begins with 'C'. To calculate sales, ignore returns and cancellations. Output the month, description of the item, and the total amount paid"""

In [0]:
spark.sql("""
    with cte as (
    select 
        stockcode,
        month(invoicedate) as month, 
        SUM(quantity*unitprice) as total_paid 
    from sales
    where invoiceno not like 'C%'
    group by month(invoicedate), stockcode),
    
    cte2 as (
    select *, rank() over (partition by month order by total_paid desc) as rnk 
    from cte)
    select * from cte2 where rnk=1
""").display()

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
df.filter(~F.col('invoiceno').startswith('C')). \
    withColumn('month', F.month(F.col('invoicedate'))). \
    groupBy('month', 'stockcode'). \
    agg(F.sum(F.col('quantity') * F.col('unitprice')).alias('revenue')). \
    withColumn('rnk', F.rank().over(Window.partitionBy('month').orderBy(F.desc('revenue')))). \
    filter(F.col('rnk') == 1) \
.display()